# PSI-159 → Google Drive (pobieranie bezpośrednie, bez dysku lokalnego)

Zbiór **Advanced Combustion via Microgravity Experiments / Cool Flames Investigation with Gases**
(PSI-159, DOI `10.60555/cyt1-kh64`, licencja CC0) — 3 756 unikalnych plików, **2,09 TB**:
987 ZIP-ów z danymi do wizualizacji płomienia (2 002 GB), 336 nagrań `.ts` (81,7 GB),
682 CSV, 93 PDF, 610 `.mdipsug`, miniatury i logi.

**Jak to działa.** Każdy plik jest pobierany przez presigned URL NASA i strumieniowany
wprost do Google Drive przez *resumable upload* — bajty nie lądują na dysku VM.
Po każdym pliku zapisywany jest postęp; wznowienie pomija pliki o zgodnym rozmiarze.

**Uruchomienie.** Komórka 1 → zaloguj się swoim kontem; komórka 2 → spis plików;
komórka 3 → start w tle; komórka 4 → podgląd. Możesz zamknąć kartę: proces żyje
w runtime Colab. Po restarcie runtime (limity Colab) powtórz 1 i 3 — nic nie pobiera
się drugi raz.

**Uwaga.** Colab ma limity czasu sesji; przy ~5–30 MB/s całość zajmuje dni. Runtime
jest bezstanowy — wznowienie jest tanie. Dyski Drive: 4,98 TiB wolne.

In [1]:
#@title 1. Autoryzacja Google (Twoje konto) { display-mode: "form" }
from google.colab import auth
auth.authenticate_user()
print("OK — poświadczenia zapisane w runtime Colab.")

OK — poświadczenia zapisane w runtime Colab.


In [2]:
#@title 2. Inwentarz PSI-159 (3756 plików, ~2.09 TB) { display-mode: "form" }
import json, os, ssl, urllib.request

MANIFEST = "/content/psi159_manifest.jsonl"
if not os.path.exists(MANIFEST):
    api = "https://psi.nasa.gov/geode-py/ws/repo/investigations/PSI-159"
    try:
        body = urllib.request.urlopen(api, timeout=120).read()
    except Exception:
        body = urllib.request.urlopen(api, timeout=120,
                                     context=ssl._create_unverified_context()).read()
    record = json.loads(body)
    rows = []
    for section in ("added", "experimentalTable", "images"):
        for item in record.get(section) or []:
            item = dict(item)
            item["section"] = section
            rows.append(item)
    with open(MANIFEST, "w", encoding="utf-8") as fh:
        for row in rows:
            fh.write(json.dumps(row) + "\n")

rows = [json.loads(l) for l in open(MANIFEST, encoding="utf-8") if l.strip()]
names = {r["file_name"] for r in rows}
total = sum((r.get("file_size") or 0) for r in rows)
print(f"{len(names)} unikalnych plików, {total/1e12:.2f} TB")

3756 unikalnych plików, 2.09 TB


In [3]:
#@title 3. Start pobierania w tle (przeżywa odłączenie notebooka) { display-mode: "form" }
import os, subprocess, textwrap

WORKER = open("/content/psi159_worker.py", "w", encoding="utf-8")
WORKER.write(textwrap.dedent(r'''
#!/usr/bin/env python3
"""Stream PSI-159 files from NASA presigned S3 URLs straight into Google Drive.

No local staging: each S3 body is read sequentially and pushed to Drive through
a resumable upload session. Progress survives interruptions; reruns skip files
whose Drive size already matches the manifest.
"""
import json
import os
import sys
import threading
import time
import urllib.parse
from concurrent.futures import ThreadPoolExecutor

import google.auth
import requests
from google.auth.transport.requests import Request as GoogleRequest

ACC = "PSI-159"
API = "https://psi.nasa.gov/geode-py/ws/studies"
ROOT_FOLDER = os.environ["PSI_ROOT_FOLDER"]
SUBFOLDER_NAME = os.environ.get("PSI_SUBFOLDER", "PSI-159_ACME_CFI-G")
CHUNK = int(os.environ.get("CHUNK", str(64 * 1024 * 1024)))
WORKERS = int(os.environ.get("WORKERS", "8"))
STATE = "/content/psi159_state.json"
LOG = "/content/psi159_downloads.log"
SCOPES = ["https://www.googleapis.com/auth/drive"]

_token_lock = threading.Lock()
try:
    _creds, _ = google.auth.default(scopes=SCOPES)
except Exception:  # Colab's cached credentials may reject explicit scopes
    _creds, _ = google.auth.default()
_log_lock = threading.Lock()


def get_token() -> str:
    with _token_lock:
        if not getattr(_creds, "valid", False):
            _creds.refresh(GoogleRequest())
        return _creds.token


def auth_headers() -> dict:
    return {"Authorization": "Bearer " + get_token()}


def session() -> requests.Session:
    s = requests.Session()
    s.headers.update({"User-Agent": "psi159-colab/1.0"})
    return s


def ensure_folder(sess: requests.Session) -> str:
    q = (f"'{ROOT_FOLDER}' in parents and name = '{SUBFOLDER_NAME}' "
         "and mimeType = 'application/vnd.google-apps.folder' and trashed = false")
    r = sess.get("https://www.googleapis.com/drive/v3/files",
                 params={"q": q, "fields": "files(id,name)", "pageSize": 10},
                 headers=auth_headers(), timeout=90)
    r.raise_for_status()
    files = r.json().get("files", [])
    if files:
        return files[0]["id"]
    r = sess.post("https://www.googleapis.com/drive/v3/files",
                  json={"name": SUBFOLDER_NAME, "mimeType": "application/vnd.google-apps.folder",
                        "parents": [ROOT_FOLDER]},
                  headers=auth_headers(), timeout=90)
    r.raise_for_status()
    return r.json()["id"]


def list_existing(sess: requests.Session, folder: str) -> dict:
    out, page = {}, None
    while True:
        params = {"q": f"'{folder}' in parents and trashed = false",
                  "fields": "nextPageToken,files(id,name,size,mimeType)",
                  "pageSize": 1000}
        if page:
            params["pageToken"] = page
        r = sess.get("https://www.googleapis.com/drive/v3/files", params=params,
                     headers=auth_headers(), timeout=120)
        r.raise_for_status()
        body = r.json()
        for item in body.get("files", []):
            if item.get("mimeType") != "application/vnd.google-apps.folder":
                out[item["name"]] = (item["id"], int(item.get("size") or 0))
        page = body.get("nextPageToken")
        if not page:
            return out


def delete_file(sess: requests.Session, file_id: str) -> None:
    sess.delete(f"https://www.googleapis.com/drive/v3/files/{file_id}",
                headers=auth_headers(), timeout=90)


def presign(sess: requests.Session, name: str, version) -> str:
    params = {"file": name, "redirect": "false"}
    if version:
        params["version"] = str(version)
    last = None
    for attempt in range(6):
        try:
            r = sess.get(f"{API}/{ACC}/download", params=params, timeout=90)
            text = r.text.strip()
            if r.ok and text.startswith("https://"):
                return text
            last = f"status={r.status_code} body={text[:200]!r}"
        except requests.RequestException as exc:
            last = repr(exc)
        time.sleep(min(30, 2 ** attempt))
    raise RuntimeError(f"presign failed for {name}: {last}")


def upload(sess: requests.Session, folder: str, name: str, expected: int, version) -> int:
    for attempt in range(3):
        url = presign(sess, name, version)
        with sess.get(url, stream=True, timeout=(30, 900)) as src:
            src.raise_for_status()
            size = expected or int(src.headers.get("Content-Length") or 0)
            init = sess.post(
                "https://www.googleapis.com/upload/drive/v3/files",
                params={"uploadType": "resumable", "fields": "id,size"},
                json={"name": name, "parents": [folder]}, headers=auth_headers(),
                timeout=120)
            init.raise_for_status()
            location = init.headers["Location"]

            raw = src.raw
            offset, eof, pending = 0, False, bytearray()
            file_id = None
            while True:
                while len(pending) < CHUNK and not eof:
                    data = raw.read(CHUNK - len(pending))
                    if not data:
                        eof = True
                        break
                    pending.extend(data)
                if not pending:
                    break
                end = offset + len(pending) - 1
                total = str(size) if size else "*"
                for _ in range(6):
                    resp = sess.put(location, data=bytes(pending),
                                    headers={"Content-Range": f"bytes {offset}-{end}/{total}",
                                             **auth_headers()},
                                    timeout=(30, 900))
                    if resp.status_code in (200, 201):
                        file_id = resp.json().get("id")
                        got = int(resp.json().get("size") or 0)
                        if file_id and size and got != size:
                            delete_file(sess, file_id)
                            raise RuntimeError(f"size mismatch {name}: drive={got} expected={size}")
                        return got
                    if resp.status_code == 308:
                        rng = resp.headers.get("Range")
                        confirmed = int(rng.rsplit("-", 1)[1]) + 1 if rng else end + 1
                        consumed = confirmed - offset
                        pending = pending[consumed:] if consumed < len(pending) else bytearray()
                        offset = confirmed
                        break
                    if resp.status_code in (401, 403, 429, 500, 502, 503, 504):
                        time.sleep(min(60, 2 ** _))
                        continue
                    raise RuntimeError(f"upload {name} failed: {resp.status_code} {resp.text[:300]}")
                else:
                    raise RuntimeError(f"upload {name}: retries exhausted at offset {offset}")
        if attempt == 2:
            raise RuntimeError(f"upload {name}: giving up after {attempt + 1} attempts")
        time.sleep(2 * (attempt + 1))


def main() -> int:
    sess = session()
    folder = ensure_folder(sess)
    rows = [json.loads(line) for line in open("/content/psi159_manifest.jsonl", encoding="utf-8") if line.strip()]
    chosen = {}
    for row in rows:
        name = row["file_name"]
        if row.get("restricted"):
            continue
        if name not in chosen or chosen[name]["section"] == "added":
            chosen[name] = row
    targets = sorted(chosen.values(), key=lambda r: r.get("file_size") or 0)

    existing = list_existing(sess, folder)
    done = {}
    if os.path.exists(STATE):
        done = json.load(open(STATE, encoding="utf-8"))
    for name, (file_id, size) in existing.items():
        done.setdefault(name, size)
    del existing

    pending = []
    for row in targets:
        want = row.get("file_size") or 0
        if done.get(row["file_name"]) == want:
            continue
        pending.append(row)
    total = sum(r.get("file_size") or 0 for r in pending)
    print(f"PSI-159: {len(targets)} files total; {len(pending)} pending "
          f"({total / 1e9:.1f} GB); {WORKERS} workers; folder={folder}", flush=True)

    qlock = threading.Lock()
    queue = list(pending)
    counter = {"files": 0, "bytes": 0}
    started = time.time()

    def worker(idx: int) -> None:
        sess = session()
        while True:
            with qlock:
                if not queue:
                    return
                row = queue.pop(0)
            name = row["file_name"]
            want = row.get("file_size") or 0
            try:
                prev = done.get(name)
                if prev is not None and prev != want:
                    ids = list_existing(sess, folder)
                    if name in ids:
                        delete_file(sess, ids[name][0])
                got = upload(sess, folder, name, want, row.get("version"))
            except Exception as exc:
                with _log_lock, open(LOG, "a", encoding="utf-8") as fh:
                    fh.write(json.dumps({"ts": time.time(), "file": name, "ok": False,
                                         "error": str(exc)[:500]}) + "\n")
                print(f"[w{idx}] FAIL {name}: {exc}", flush=True)
                continue
            with qlock:
                done[name] = got
                json.dump(done, open(STATE, "w", encoding="utf-8"))
                counter["files"] += 1
                counter["bytes"] += got
                with open(LOG, "a", encoding="utf-8") as fh:
                    fh.write(json.dumps({"ts": time.time(), "file": name, "ok": True,
                                         "size": got}) + "\n")
                elapsed = max(1e-9, time.time() - started)
                rate = counter["bytes"] / elapsed / 1e6
                remain = (total - counter["bytes"]) / max(1e-9, counter["bytes"] / elapsed) / 3600
                print(f"[w{idx}] {counter['files']}/{len(pending)} {name} "
                      f"{got / 1e6:.1f} MB | {rate:.1f} MB/s | ETA {remain:.1f} h", flush=True)

    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        list(pool.map(worker, range(WORKERS)))

    missing = [r["file_name"] for r in targets if done.get(r["file_name"]) != (r.get("file_size") or 0)]
    print(f"finished: {counter['files']} uploaded, {counter['bytes'] / 1e9:.2f} GB in "
          f"{(time.time() - started) / 3600:.2f} h; incomplete={len(missing)}", flush=True)
    return 0 if not missing else 1


if __name__ == "__main__":
    sys.exit(main())
'''))
WORKER.close()

env = dict(os.environ, PSI_ROOT_FOLDER="1oKAVMMAEP_YsZ_REAoAt8YZV6X5E2gk1",
           WORKERS="8", CHUNK=str(64 * 1024 * 1024))
subprocess.run(["pkill", "-f", "psi159_worker.py"], check=False)
with open("/content/psi159.log", "a") as log:
    subprocess.Popen(["python3", "/content/psi159_worker.py"], env=env,
                     stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print("Uruchomiono w tle. Log: /content/psi159.log")

Uruchomiono w tle. Log: /content/psi159.log


In [24]:
#@title 4. Podgląd postępu { display-mode: "form" }
!tail -n 5 /content/psi159.log
import json, os
manifest = [json.loads(l) for l in open("/content/psi159_manifest.jsonl", encoding="utf-8") if l.strip()]
names = {r["file_name"]: (r.get("file_size") or 0) for r in manifest}
state = json.load(open("/content/psi159_state.json")) if os.path.exists("/content/psi159_state.json") else {}
ok = sum(1 for k, v in state.items() if names.get(k) == v)
print(f"\nGotowe: {ok}/{len(names)} plików, {sum(state.values())/1e9:.1f} GB z {sum(names.values())/1e9:.1f} GB")

[w2] 1785/3720 PSI-159_Raw Data_INTNS_21130A7.mdipsug 0.3 MB | 0.1 MB/s | ETA 5504.4 h
[w1] 1786/3720 PSI-159_Raw Data_INTNS_21106G7.mdipsug 0.3 MB | 0.1 MB/s | ETA 5478.3 h
[w6] 1787/3720 PSI-159_Raw Data_INTNS_21130G4.mdipsug 0.3 MB | 0.1 MB/s | ETA 5452.8 h
[w7] 1788/3720 PSI-159_Raw Data_INTNS_21222A2.mdipsug 0.3 MB | 0.1 MB/s | ETA 5428.2 h
[w3] 1789/3720 PSI-159_Raw Data_INTNS_21265K1.mdipsug 0.3 MB | 0.1 MB/s | ETA 5404.5 h

Gotowe: 1825/3756 plików, 0.2 GB z 2085.6 GB
